# Train model kiểm duyệt bình luận cho TikTak (PhoBERT-v2 + ViHSD)

Fine-tune `vinai/phobert-base-v2` phân loại bình luận tiếng Việt thành **3 nhãn**:

| Nhãn | Ý nghĩa |
|---|---|
| `clean` | bình thường (kể cả chê, buồn, góp ý) |
| `offensive` | tục tĩu, xúc phạm |
| `hate` | thù ghét, công kích nhóm người |

`ai-service` của TikTak tính **điểm vi phạm = 1 − P(clean)** rồi chia 3 mức: hiện ngay / chờ admin duyệt / ẩn luôn.

### Trước khi chạy (Kaggle)
1. **Settings → Accelerator: GPU T4 x2** (tránh P100: bản PyTorch mới trên Kaggle không còn hỗ trợ).
2. **Settings → Internet: On** (để tải dataset, PhoBERT và đẩy model lên HuggingFace).
3. **Add-ons → Secrets → Add Secret**: ô **Label** gõ đúng chữ `HF_TOKEN`, ô **Value** dán token `hf_...` quyền **Write** (tạo ở https://huggingface.co/settings/tokens), rồi bật công tắc cho notebook này.
4. Sửa `HUB_REPO` ở ô cấu hình bên dưới thành tên repo của bạn.
5. **Run All**. Train mất khoảng 20–30 phút trên T4.

Chạy xong: thêm `AI_MODERATION_MODEL_PATH=<HUB_REPO>` vào file `.env` cạnh `docker-compose.yml` (repo private thì thêm `HF_TOKEN=...` quyền Read), rồi `docker compose up -d --build ai-service`.

In [ ]:
!pip install -q pyvi

## 1. Cấu hình

In [ ]:
HUB_REPO   = "hoangkkk/phobert-v2-vihsd-toxic"   # ← đổi thành tên-bạn/tên-repo
PRIVATE    = False          # True = repo riêng tư (ai-service cần HF_TOKEN để tải)
BASE_MODEL = "vinai/phobert-base-v2"

MAX_LEN    = 128            # bình luận ngắn; PhoBERT tối đa 256
EPOCHS     = 5
BATCH_SIZE = 32
LR         = 2e-5
WARMUP     = 0.1            # tỉ lệ bước warmup
PATIENCE   = 2              # dừng sớm nếu macro-F1 trên tập validation không tăng sau N epoch
SEED       = 42

# Thêm dữ liệu của bạn (không bắt buộc): CSV 2 cột `text,label` với label ∈ clean|offensive|hate.
# Upload lên Kaggle (Add Input) rồi điền đường dẫn, VD "/kaggle/input/binh-luan-tiktak/extra.csv"
EXTRA_CSV  = None

LABELS = ["clean", "offensive", "hate"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

In [ ]:
import os, re, random, unicodedata, glob, json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Dataset

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
if device.type != "cuda":
    print("⚠️  Không có GPU — bật Settings → Accelerator → GPU T4 x2, không thì train rất lâu.")

## 2. Tải dữ liệu ViHSD

ViHSD (UIT, ~33K bình luận Facebook/YouTube, giấy phép MIT). Notebook thử lần lượt các bản trên HuggingFace;
nếu đều lỗi thì tìm file CSV bạn đã Add Input trên Kaggle.
Mã nhãn số của bản gốc: **0 = CLEAN, 1 = OFFENSIVE, 2 = HATE**.

In [ ]:
from datasets import load_dataset

NUM2NAME = {0: "clean", 1: "offensive", 2: "hate"}
STR2NAME = {"clean": "clean", "offensive": "offensive", "hate": "hate"}

def to_name(v):
    if isinstance(v, (int, np.integer)) or (isinstance(v, str) and v.strip().isdigit()):
        return NUM2NAME[int(v)]
    return STR2NAME[str(v).strip().lower()]

def pick(df, cands):
    for c in cands:
        if c in df.columns: return c
    raise KeyError(f"Không thấy cột nào trong {cands}; có: {list(df.columns)}")

def normalize_frame(df):
    t = pick(df, ["free_text", "text", "comment", "content"])
    l = pick(df, ["label_id", "label", "labels"])
    out = pd.DataFrame({"text": df[t].astype(str), "label": df[l].map(to_name)})
    return out.dropna()

def from_phucdev():
    ds = load_dataset("phucdev/ViHSD")
    return {s: normalize_frame(ds[s].to_pandas()) for s in ["train", "validation", "test"]}

def from_visolex():
    df = load_dataset("visolex/ViHSD")["train"].to_pandas()
    split = pick(df, ["type", "split"])
    m = {"train": "train", "dev": "validation", "validation": "validation", "valid": "validation", "test": "test"}
    df["_s"] = df[split].str.lower().map(m)
    return {s: normalize_frame(df[df["_s"] == s]) for s in ["train", "validation", "test"]}

def from_kaggle_input():
    files = {os.path.basename(p).lower(): p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)}
    def find(*names):
        for n in names:
            for f, p in files.items():
                if n in f: return p
    tr, dv, te = find("train"), find("dev", "valid"), find("test")
    if not (tr and dv and te): raise FileNotFoundError(f"Không thấy đủ train/dev/test trong /kaggle/input: {list(files)}")
    return {"train": normalize_frame(pd.read_csv(tr)), "validation": normalize_frame(pd.read_csv(dv)), "test": normalize_frame(pd.read_csv(te))}

data = None
for name, fn in [("phucdev/ViHSD", from_phucdev), ("visolex/ViHSD", from_visolex), ("Kaggle input", from_kaggle_input)]:
    try:
        data = fn(); print("✓ Dữ liệu từ", name); break
    except Exception as e:
        print("✗", name, "→", type(e).__name__, str(e)[:150])
assert data is not None, "Không tải được ViHSD — bật Internet, hoặc Add Input dataset ViHSD (train/dev/test.csv)"

if EXTRA_CSV:
    extra = normalize_frame(pd.read_csv(EXTRA_CSV))
    data["train"] = pd.concat([data["train"], extra], ignore_index=True)
    print(f"+ {len(extra)} dòng từ {EXTRA_CSV}")

for s, df in data.items():
    print(f"{s:10s} {len(df):6d} dòng ", df["label"].value_counts().to_dict())

In [ ]:
# Kiểm tra bằng mắt: nhãn có khớp nội dung không (clean phải là câu bình thường)
for lab in LABELS:
    print(f"\n── {lab} ──")
    for t in data["train"][data["train"]["label"] == lab]["text"].sample(4, random_state=1):
        print("  •", t[:120])

## 3. Tiền xử lý — PHẢI giống hệt lúc chạy trong `ai-service`

PhoBERT học trên câu **đã tách từ** (`cảm_ơn`, `óc_chó`). Dùng `pyvi` để tách từ.
Tên bộ tách từ được ghi vào `config.json` (`word_segmenter: "pyvi"`) để `ai-service` tự làm đúng như vậy.

In [ ]:
from pyvi import ViTokenizer

URL_RE = re.compile(r"https?://\S+")
SPACE_RE = re.compile(r"\s+")

def preprocess(text: str) -> str:
    # Giống app/moderation.py::_preprocess trong ai-service
    text = unicodedata.normalize("NFC", str(text or ""))
    text = URL_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return ViTokenizer.tokenize(text) if text else text

t0 = time.time()
for s in data:
    data[s]["seg"] = data[s]["text"].map(preprocess)
    data[s]["y"] = data[s]["label"].map(label2id)
print(f"Tách từ xong trong {time.time()-t0:.0f}s")
print(data["train"][["text", "seg"]].head(3).to_string())

## 4. Tokenizer & DataLoader

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class CommentDS(Dataset):
    def __init__(self, df): self.x, self.y = df["seg"].tolist(), df["y"].tolist()
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

def collate(batch):
    texts, ys = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    enc["labels"] = torch.tensor(ys)
    return enc

loaders = {
    s: DataLoader(CommentDS(data[s]), batch_size=BATCH_SIZE if s == "train" else BATCH_SIZE * 2,
                  shuffle=(s == "train"), collate_fn=collate, num_workers=2)
    for s in data
}
lens = data["train"]["seg"].map(lambda t: len(tokenizer.tokenize(t)))
print(f"Số token: trung vị {lens.median():.0f}, 95% ≤ {lens.quantile(.95):.0f}, >{MAX_LEN}: {(lens > MAX_LEN).mean():.1%} bị cắt")

## 5. Model + trọng số lớp

Dữ liệu lệch (khoảng 80% `clean`). Trọng số lớp = căn bậc hai nghịch đảo tần suất: giúp bắt được câu vi phạm
mà không làm model "thấy đâu cũng vi phạm".

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id
).to(device)
if torch.cuda.device_count() > 1:
    print("Dùng", torch.cuda.device_count(), "GPU")
    model = torch.nn.DataParallel(model)

counts = data["train"]["y"].value_counts().sort_index().values.astype(float)
weights = torch.tensor((counts.sum() / counts) ** 0.5, dtype=torch.float)
weights = (weights / weights.mean()).to(device)
print("Trọng số lớp:", dict(zip(LABELS, weights.round(decimals=2).tolist())))
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

## 6. Train

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

def unwrap(m): return m.module if hasattr(m, "module") else m

@torch.no_grad()
def predict(loader):
    model.eval(); probs, ys = [], []
    for b in loader:
        ys += b.pop("labels").tolist()
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(**b).logits
        probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy(), np.array(ys)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = len(loaders["train"]) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

best_f1, best_state, bad = -1, None, 0
for ep in range(1, EPOCHS + 1):
    model.train(); total = 0
    for b in tqdm(loaders["train"], desc=f"Epoch {ep}/{EPOCHS}", leave=False):
        y = b.pop("labels").to(device)
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            loss = loss_fn(model(**b).logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    p, y = predict(loaders["validation"])
    f1 = f1_score(y, p.argmax(1), average="macro")
    print(f"Epoch {ep}: loss {total/len(loaders['train']):.4f} · val macro-F1 {f1:.4f}")
    if f1 > best_f1:
        best_f1, bad = f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE: print("Dừng sớm"); break

unwrap(model).load_state_dict(best_state)
print(f"\nGiữ model tốt nhất: val macro-F1 = {best_f1:.4f}")

## 7. Đánh giá trên tập test

In [ ]:
p_test, y_test = predict(loaders["test"])
print(classification_report(y_test, p_test.argmax(1), target_names=LABELS, digits=3))
print("Ma trận nhầm lẫn (hàng = thật, cột = đoán):")
print(pd.DataFrame(confusion_matrix(y_test, p_test.argmax(1)), index=LABELS, columns=LABELS))

## 8. Chọn ngưỡng cho 3 mức của TikTak

Điểm vi phạm = 1 − P(clean). Backend:
- điểm < `COMMENT_REVIEW_THRESHOLD` → hiện ngay
- giữa 2 ngưỡng → chờ admin duyệt
- điểm ≥ `COMMENT_REJECT_THRESHOLD` → ẩn luôn

Bảng dưới cho biết với mỗi ngưỡng: bao nhiêu câu **sạch bị ẩn nhầm** (nên rất thấp) và bao nhiêu câu **vi phạm lọt qua**.

In [ ]:
score = 1 - p_test[:, label2id["clean"]]
toxic = y_test != label2id["clean"]
rows = []
for t in np.arange(0.3, 0.96, 0.05):
    flagged = score >= t
    rows.append({
        "ngưỡng": round(t, 2),
        "câu sạch bị chặn": f"{(flagged & ~toxic).sum() / (~toxic).sum():.1%}",
        "vi phạm bắt được": f"{(flagged & toxic).sum() / toxic.sum():.1%}",
        "độ chính xác khi chặn": f"{(flagged & toxic).sum() / max(flagged.sum(), 1):.1%}",
    })
display(pd.DataFrame(rows))

def first(cond):
    for t in np.arange(0.30, 0.991, 0.01):
        if cond(t): return round(float(t), 2)
    return 0.99
# Ẩn luôn: ngưỡng thấp nhất mà ≥ 95% câu bị ẩn là vi phạm thật
reject = first(lambda t: (score >= t).sum() > 0 and ((score >= t) & toxic).sum() / (score >= t).sum() >= 0.95)
# Chờ duyệt: ngưỡng cao nhất vẫn bắt được ≥ 85% câu vi phạm (không thấp hơn 0.3, không cao hơn ngưỡng ẩn)
review = max([round(float(t), 2) for t in np.arange(0.30, reject, 0.01) if ((score >= t) & toxic).sum() / toxic.sum() >= 0.85] or [0.3])
q = ((score >= review) & (score < reject)).mean()
print(f"\nĐề xuất cho backend/.env:\n  COMMENT_REVIEW_THRESHOLD={review}\n  COMMENT_REJECT_THRESHOLD={reject}")
print(f"→ khoảng {q:.1%} bình luận sẽ vào hàng chờ duyệt, {(score >= reject).mean():.1%} bị ẩn luôn")

## 9. Thử vài câu (gồm câu chê nhưng không vi phạm)

In [ ]:
@torch.no_grad()
def check(texts):
    model.eval()
    enc = tokenizer([preprocess(t) for t in texts], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    p = torch.softmax(model(**enc).logits.float(), -1).cpu().numpy()
    for t, row in zip(texts, p):
        s = 1 - row[0]
        lvl = "ẨN LUÔN " if s >= reject else "CHỜ DUYỆT" if s >= review else "HIỆN NGAY"
        print(f"{lvl}  {s:.3f}  {dict(zip(LABELS, row.round(3)))}  {t}")

check([
    "Video hay quá, cảm ơn bạn!",
    "Quay ở đâu vậy bạn ơi?",
    "Phim này dở quá, phí thời gian",
    "Tiếc ghê, hôm đó mình không đi được",
    "Nhìn ngu vãi",
    "Đồ óc chó, cút đi",
    "đm quay xấu vcl",
    "Bọn bắc kỳ/nam kỳ toàn lũ lừa đảo",
])

## 10. Lưu và đẩy lên HuggingFace

In [ ]:
from huggingface_hub import login, HfApi

# Token lấy từ Kaggle Secret có NHÃN (Label) là HF_TOKEN — giá trị (Value) là token hf_... quyền Write
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Không đọc được Secret HF_TOKEN. Vào Add-ons → Secrets: Label = HF_TOKEN, Value = token hf_..., "
        "bật công tắc cho notebook này rồi chạy lại ô này (không cần train lại)."
    ) from e
login(token=HF_TOKEN)

m = unwrap(model)
# Thông tin cho ai-service. KHÔNG dùng tên "max_length": transformers coi đó là tham số sinh văn bản và báo lỗi khi lưu.
if "max_length" in m.config.__dict__:
    delattr(m.config, "max_length")
m.config.word_segmenter = "pyvi"          # ai-service đọc để tách từ giống lúc train
m.config.clean_labels = ["clean"]
m.config.train_max_tokens = MAX_LEN
m.config.recommended_thresholds = {"review": review, "reject": reject}

OUT = "/kaggle/working/model"
m.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

rep = classification_report(y_test, p_test.argmax(1), target_names=LABELS, digits=3)
with open(f"{OUT}/README.md", "w", encoding="utf-8") as f:
    f.write(f'''---
language: vi
license: mit
base_model: {BASE_MODEL}
datasets: [uitnlp/vihsd]
pipeline_tag: text-classification
---
# Phân loại bình luận vi phạm (TikTak)

Fine-tune `{BASE_MODEL}` trên ViHSD. Nhãn: `clean`, `offensive`, `hate`.

**Đầu vào phải tách từ bằng pyvi** (`ViTokenizer.tokenize`) sau khi chuẩn hoá NFC, bỏ URL, gộp khoảng trắng.

Ngưỡng đề xuất (điểm = 1 − P(clean)): chờ duyệt ≥ {review}, ẩn ≥ {reject}.

## Kết quả trên tập test ViHSD
```
{rep}
```
''')

api = HfApi(token=HF_TOKEN)
api.create_repo(HUB_REPO, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=HUB_REPO, commit_message="Train trên ViHSD")
print(f"✓ Đã đẩy lên https://huggingface.co/{HUB_REPO}")
print(sorted(os.listdir(OUT)))

## 11. Gắn vào TikTak

File `.env` cạnh `docker-compose.yml`:
```
AI_MODERATION_MODEL_PATH=hoangkkk/phobert-v2-vihsd-toxic
# HF_TOKEN=hf_...      ← chỉ cần nếu PRIVATE = True (token quyền Read)
```
`backend/.env`: dán 2 dòng `COMMENT_REVIEW_THRESHOLD` / `COMMENT_REJECT_THRESHOLD` in ở bước 8.

Rồi:
```
docker compose up -d --build ai-service
docker compose exec ai-service python scripts/check_moderation.py
```
Trang Quản trị → Bình luận sẽ báo "Đang chấm bằng model NLP".

Nếu không đẩy lên Hub được: tải thư mục `/kaggle/working/model` (tab Output) và chép vào `ai-service/moderation_model/`.